In [ ]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import time
from experimental_pose_encoder_model_extension.advanced_pose_encoder import AdvancedPoseEncoder
from dataset.dataset import *
import utils.utils as utils
from torch.utils.data import DataLoader
from tqdm import tqdm
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
from torch.amp import autocast

device = utils.get_device()

In [ ]:
animation_visualisation.init_visualization()

In [ ]:
dataset = GPUDataset(
    consolidated_file="dataset/genea2023_dataset/val/main-agent/advanced_encoder/consolidated.npz",
    seq_length=100, # For testing
    seed_length=0,
    batch_size=1,
    epoch_length=100,
    loading_encoded_data = True
)

with autocast(device_type=device.type, dtype=torch.bfloat16):
    autoencoder_model = AdvancedPoseEncoder.load_from_checkpoint("advanced_pose_encoder_ik_pca_64")

    data_loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )

    progress_bar = tqdm(data_loader, desc=f'Testing', leave=True)
    for i, batch_data in enumerate(progress_bar):
        with torch.no_grad():
            
            gesture_sequence, _, audio_features, main_agent_id_one_hot, finger_availability = [
                item.squeeze(0).to(device) for item in batch_data
            ]

            if dataset.loading_encoded_data:
                gesture_sequence = autoencoder_model.decode(gesture_sequence)

            gesture_sequence = dataset.skeleton.denormalize_poses(gesture_sequence)

            # Send every frame to the visualizer
            for frame in gesture_sequence.squeeze(0):
                animation_visualisation.send_pose(frame.cpu(),dataset.skeleton)
                time.sleep(1.0 / 30.0)  # Assuming 30 FPS
                